# Track 6 example — How many people are playing right now?
### EP0418 · Free-form starter · Steam concurrent players

Steam publishes, for every game, the number of people playing at this moment.
Only the current number: there is no history to download. So the course collects
it, once an hour, for ten popular games, into `steam_players_hourly.csv`. The
file grows every week of the semester.

Why it is good data: it has a daily cycle (evenings in each time zone), a weekly
cycle (weekends), spikes on patch days and free weekends, and occasional holes
when the collector or Steam was down. Those are exactly the things Labs 2 and 4
teach you to find.

> **First: File → Save a copy in Drive, and work in your copy.** This page opened from GitHub, and changes here are not saved. Reopen your copy from Google Drive next time.

In [ ]:
import os, urllib.request, json
for _f in ["steam_players_hourly.csv"]:
    if not os.path.exists(_f):
        urllib.request.urlretrieve("https://raw.githubusercontent.com/StanleySP-eng/EP0418-labs/main/starter_packs/track6_free_form/" + _f, _f)

import pandas as pd, numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv("steam_players_hourly.csv", parse_dates=["timestamp"])
print(f"{len(df):,} rows, {df.timestamp.min()} to {df.timestamp.max()} UTC, "
      f"{df.game.nunique()} games, {df.timestamp.nunique()} hourly samples")
print(df.groupby("game").players.agg(["count", "min", "median", "max"]).sort_values("median", ascending=False))

### The live number

The cell below asks Steam for the count right now. Compare it with the latest row
in the file: that is the simplest possible check that the file is real.

In [ ]:
APPID = 730      # Counter-Strike 2. Dota 2 is 570; see the file for the others.
url = f"https://api.steampowered.com/ISteamUserStats/GetNumberOfCurrentPlayers/v1/?appid={APPID}"
try:
    with urllib.request.urlopen(url, timeout=30) as r:
        live = json.load(r)["response"]["player_count"]
    print(f"playing now (appid {APPID}): {live:,}")
except Exception as e:
    print("Steam did not answer:", e)
last = df[df.appid == APPID].sort_values("timestamp").iloc[-1]
print(f"last row in the file: {last.players:,.0f} at {last.timestamp} UTC")

### One game, on a regular hourly grid

Singapore is UTC+8. Converting the time zone before you look for the daily cycle
is the first decision, and it is the kind of thing an AI assistant forgets.

In [ ]:
GAME = df[df.appid == APPID].set_index("timestamp").players
GAME.index = GAME.index.tz_localize("UTC").tz_convert("Asia/Singapore")
s = GAME.resample("h").mean()                     # regular grid; gaps stay as NaN
print(f"{len(s)} hourly slots, {s.isna().sum()} empty")

fig, axes = plt.subplots(2, 1, figsize=(13, 7))
s.plot(ax=axes[0], title=f"appid {APPID}: players online, Singapore time")
if s.notna().sum() >= 48:
    s.groupby(s.index.hour).mean().plot(ax=axes[1], marker="o",
                                         title="mean by hour of day (Singapore time)")
else:
    axes[1].text(0.5, 0.5, "Fewer than two days collected so far; the hour-of-day plot needs more data.",
                 ha="center", va="center"); axes[1].set_axis_off()
plt.tight_layout(); plt.show()

### The baseline: same hour last week

Seasonal-naive, as in the taxi data. It needs at least two weeks of rows. Until
then the cell says so rather than inventing a number.

In [ ]:
if s.notna().sum() >= 24 * 14:
    snaive = s.shift(168)
    m = s.notna() & snaive.notna()
    mae = (s[m] - snaive[m]).abs().mean()
    print(f"seasonal-naive (same hour last week) MAE: {mae:,.0f} players, over {m.sum()} hours")
    print(">>> That is the number a forecast has to beat.")
else:
    print(f"Only {s.notna().sum()} hours collected so far. The weekly baseline needs 336. Come back later.")

---
## Three macro questions to shape a proposal

**1. What decision does a player count drive?**
A server operator deciding when to run maintenance; a tournament organiser
picking a start time; a streamer choosing when to go live. Pick one person and
one threshold. Without them you have a chart, not a system (SLO 7.2).

**2. What is an anomaly here, and what is just Saturday?**
A patch day looks like an anomaly to a z-score and like a normal Tuesday to a
seasonal model. Decide which detector from Lab 4 fits, and check its flags against
the game's own patch notes, which are public (SLO 5.1, 5.3).

**3. How does a missing hour differ from a quiet hour?**
The collector sometimes fails and leaves a hole. Steam sometimes goes down and
reports a tiny number. Your cleaning step must tell these apart before any
forecast sees them (SLO 4.1).